# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR²) Exploration with `mlcroissant`
This notebook provides a walkthrough for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, referencing all dataset entities via their unique `@id` fields. This facilitates reproducible and robust data workflows.

### Dataset Source
The dataset is defined by a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed. Remove the '!' if running outside Jupyter notebooks.
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")


## 2. Data Overview
Review available record sets and their fields, referencing all by their `@id`.

In [ ]:
# List all record sets (`cr:RecordSet`) and fields
print("Available record sets in the dataset:\n")
recset_list = []
for recset in metadata.record_sets:
    print(f"- {recset['@id']}: {recset.get('name','[no name]')}")
    recset_list.append(recset['@id'])
    # List fields for each record set
    if 'fields' in recset:
        print("  Fields:")
        for f in recset['fields']:
            print(f"    - {f['@id']}: {f.get('name','[no name]')} (type: {f.get('dataType','n/a')})")
    print()
if not recset_list:
    print('(No record sets found in metadata. Data may be defined in datafiles only.)')


## 3. Data Extraction
Extract data from a main record set using its `@id`, and load the records into a DataFrame. Reference fields by their `@id`.

In [ ]:
# Identify the main record set @id (manual inspection after previous cell)
# For this dataset, there is typically a main record set (e.g., cr:RecordSet/Subjects or similar).

# Workaround for Croissant Datasets that may define record sets only in files:
# Try to print all available record sets declared in the metadata, otherwise use 'dataset.record_sets()' directly:

record_sets = [recset['@id'] for recset in getattr(metadata, 'record_sets', [])]

if not record_sets:
    # Fallback: try to fetch from dataset interface (mlcroissant supports this)
    record_sets = list(dataset.record_sets())

print(f"Detected record set @id(s): {record_sets}")

dataframes = {}
for record_set_id in record_sets:
    print(f"\nLoading record set: {record_set_id}")
    rows = list(dataset.records(record_set=record_set_id))
    if rows:
        df = pd.DataFrame(rows)
        print(f"Columns for {record_set_id}: {list(df.columns)}\nFirst 5 rows:")
        print(df.head())
        dataframes[record_set_id] = df
    else:
        print("No records found for this record set.")

# Pick the first record set as the main one for further analysis
if record_sets:
    main_record_set_id = record_sets[0]
    main_df = dataframes.get(main_record_set_id)
else:
    main_record_set_id = None
    main_df = None

if main_df is not None:
    print(f"\nMain DataFrame ({main_record_set_id}) has shape: {main_df.shape}")
else:
    print("No data extracted.")


## 4. Exploratory Data Analysis (EDA)
Apply analytic steps such as filtering, normalization, and groupby, referencing fields by their `@id`.

⚠️ If you aren't sure of column `@id`s, check the column names printed above and match them to the schema.

In [ ]:
if main_df is not None and len(main_df) > 0:
    # Try to detect a numeric field, such as 'age', 'interval_months', or similar by inspecting columns
    candidate_numeric_ids = [col for col in main_df.columns if ('age' in col.lower() or 'interval' in col.lower() or 'month' in col.lower())]
    
    if candidate_numeric_ids:
        numeric_field = candidate_numeric_ids[0]  # Use the first matching candidate
    else:
        # Fallback to first numeric column
        numeric_field = next((col for col in main_df.columns if pd.api.types.is_numeric_dtype(main_df[col])), None)
    
    if numeric_field is not None:
        print(f"Using numeric field '{numeric_field}' for EDA.")
        threshold = main_df[numeric_field].mean() # For demonstration, use mean as threshold
        filtered_df = main_df[main_df[numeric_field] > threshold].copy()
        print(f"\nFiltered records with {numeric_field} > {threshold:.2f} (threshold: mean):")
        print(filtered_df[[numeric_field]].head())
        
        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        
        # Try grouping by a categorical field (e.g., 'sex', 'cancer_type', etc)
        group_candidates = [col for col in main_df.columns if ('sex' in col.lower() or 'type' in col.lower() or 'location' in col.lower() or 'group' in col.lower()) and col != numeric_field]
        group_field = group_candidates[0] if group_candidates else None
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(f'mean_{numeric_field}')
            print(f"\nGrouped data (mean {numeric_field}) by {group_field}:")
            print(grouped_df.head())
        else:
            print("\nNo suitable group field found.")
    else:
        print("No numeric fields identified in main DataFrame.")   
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize the distribution of a numeric field and an optional group comparison.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_df is not None and main_df.shape[1] > 0:
    field_to_plot = None
    if 'numeric_field' in locals() and numeric_field and numeric_field in main_df.columns:
        field_to_plot = numeric_field
    else:
        nums = [col for col in main_df.columns if pd.api.types.is_numeric_dtype(main_df[col])]
        if nums:
            field_to_plot = nums[0]
    
    if field_to_plot:
        plt.figure(figsize=(8,4))
        sns.histplot(main_df[field_to_plot], kde=True, bins=15)
        plt.title(f"Distribution of {field_to_plot}")
        plt.xlabel(field_to_plot)
        plt.ylabel("Count")
        plt.show()

        # If a group field exists, show boxplot
        if 'group_field' in locals() and group_field and group_field in main_df.columns:
            plt.figure(figsize=(10,5))
            sns.boxplot(data=main_df, x=group_field, y=field_to_plot)
            plt.title(f"{field_to_plot} by {group_field}")
            plt.xlabel(group_field)
            plt.ylabel(field_to_plot)
            plt.xticks(rotation=30)
            plt.show()
    else:
        print("No suitable numeric field to plot.")
else:
    print("No data to visualize.")

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² dataset using the `mlcroissant` library, referencing record sets and fields explicitly by their `@id` as required by FAIR data practices.

- We reviewed record sets and fields present in the dataset.
- Extracted data with all entity references via `@id`.
- Applied numeric filtering, normalization, and group analysis.
- Visualized distributions of numeric variables for further insight.

This notebook provides a template for robust and reproducible exploration of Croissant datasets using entity `@id` references and can be easily extended for more advanced analyses or for integrating additional Croissant-compliant datasets.